## **Goal**
**Generate a Pandas Dataframe with:**
* id (e.g., 00001)
* camera_pitch (deg)
* distance (cm)
* angular_separation (deg)
* delta_yaw (deg)
* delta_pitch (deg)
* vec_x (float)
* vec_y (float)
* vec_z (float)

In [29]:
import os
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

In [30]:
# Constants

NUM_ROWS = 4096

DATASET_PATH = "../../../dataset"
IMAGES_A_PATH = os.path.join(DATASET_PATH, 'images_A')
IMAGES_B_PATH = os.path.join(DATASET_PATH, 'images_B')
METADATA_PATH = os.path.join(DATASET_PATH, 'metadata_A')

DATAFRAME_PATH = "../../eval"
SAVE_PATH = os.path.join(DATAFRAME_PATH, "pipeline_results.csv")

FOCAL_LENGTH_MM = 35.0
SENSOR_WIDTH_MM = 36.0 
IMAGE_WIDTH_PX = 1920
BASELINE_M = 0.1

In [31]:
# Structure

pipelines = ['gt', 'b3ad', 'b3sd', 'm3ad', 'm3sd', 'm2as', 'm2st', 'm2pa']
metrics = ['dist', 'ang_sep', 'd_yaw', 'd_pitch', 'sw_x', 'sw_y', 'sw_z', 'sc_x', 'sc_y', 'sc_z']

In [32]:
# Column List

cols = ['id', 'actor', 'pose', 'cam_pitch']
for p in pipelines:
    for m in metrics:
        cols.append(f'{p}_{m}')
print(cols)

['id', 'actor', 'pose', 'cam_pitch', 'gt_dist', 'gt_ang_sep', 'gt_d_yaw', 'gt_d_pitch', 'gt_sw_x', 'gt_sw_y', 'gt_sw_z', 'gt_sc_x', 'gt_sc_y', 'gt_sc_z', 'b3ad_dist', 'b3ad_ang_sep', 'b3ad_d_yaw', 'b3ad_d_pitch', 'b3ad_sw_x', 'b3ad_sw_y', 'b3ad_sw_z', 'b3ad_sc_x', 'b3ad_sc_y', 'b3ad_sc_z', 'b3sd_dist', 'b3sd_ang_sep', 'b3sd_d_yaw', 'b3sd_d_pitch', 'b3sd_sw_x', 'b3sd_sw_y', 'b3sd_sw_z', 'b3sd_sc_x', 'b3sd_sc_y', 'b3sd_sc_z', 'm3ad_dist', 'm3ad_ang_sep', 'm3ad_d_yaw', 'm3ad_d_pitch', 'm3ad_sw_x', 'm3ad_sw_y', 'm3ad_sw_z', 'm3ad_sc_x', 'm3ad_sc_y', 'm3ad_sc_z', 'm3sd_dist', 'm3sd_ang_sep', 'm3sd_d_yaw', 'm3sd_d_pitch', 'm3sd_sw_x', 'm3sd_sw_y', 'm3sd_sw_z', 'm3sd_sc_x', 'm3sd_sc_y', 'm3sd_sc_z', 'm2as_dist', 'm2as_ang_sep', 'm2as_d_yaw', 'm2as_d_pitch', 'm2as_sw_x', 'm2as_sw_y', 'm2as_sw_z', 'm2as_sc_x', 'm2as_sc_y', 'm2as_sc_z', 'm2st_dist', 'm2st_ang_sep', 'm2st_d_yaw', 'm2st_d_pitch', 'm2st_sw_x', 'm2st_sw_y', 'm2st_sw_z', 'm2st_sc_x', 'm2st_sc_y', 'm2st_sc_z', 'm2pa_dist', 'm2pa_ang_s

In [33]:
# Initialize Dataframe

df = pd.DataFrame(np.nan, index=range(NUM_ROWS), columns=cols)
df['id'] = df['id'].astype(str)
df['actor'] = df['actor'].astype(str)

In [34]:
# ID Column

df['id'] = [f"{i:05d}" for i in range(1, NUM_ROWS + 1)]
print(df[['id']].head())

      id
0  00001
1  00002
2  00003
3  00004
4  00005


In [35]:
def gt_calcs(id):
    
    # Load JSON Metadata
    json_path = F"{METADATA_PATH}/{id}A.json"
    with open(json_path, 'r') as file:
        raw_data = json.load(file)
    
    # Other Metadata
    actor = raw_data['ID']
    pose = raw_data['PoseIndex']

    # Keypoint Global Coordinates
    shoulder_coords_global = np.array([raw_data['Right Shoulder Coords'][axis] for axis in ['x', 'y', 'z']])
    wrist_coords_global = np.array([raw_data['Right Wrist Coords'][axis] for axis in ['x', 'y', 'z']])
    camera_coords_global = np.array([raw_data['Camera Coords'][axis] for axis in ['x', 'y', 'z']])
    camera_forward_global = np.array([raw_data['Camera Rotator'][axis] for axis in ['x', 'y', 'z']])

    # Assumed Constants
    camera_coords = np.array([0, 0, 0])
    world_up = np.array([0, 0, 1])

    # Create Rotation Matrix
    camera_right_global = np.cross(world_up, camera_forward_global) 
    camera_right_global /= np.linalg.norm(camera_right_global)

    camera_up_global = np.cross(camera_forward_global, camera_right_global)
    camera_up_global /= np.linalg.norm(camera_up_global)

    rotation_matrix = np.vstack((camera_forward_global,
                                camera_right_global,
                                camera_up_global))

    # Translate To Origin
    shoulder_coords = shoulder_coords_global - camera_coords_global
    wrist_coords = wrist_coords_global - camera_coords_global

    # Apply Rotation Transformation
    shoulder_coords = rotation_matrix @ shoulder_coords
    wrist_coords = rotation_matrix @ wrist_coords

    # Shoulder-Camera Distance
    distance = np.linalg.norm(shoulder_coords)

    # Shoulder-Wrist Shoulder-Camera Vectors
    shoulder_wrist = wrist_coords - shoulder_coords
    shoulder_wrist /= np.linalg.norm(shoulder_wrist)

    shoulder_camera = camera_coords - shoulder_coords
    shoulder_camera /= np.linalg.norm(shoulder_camera)

    # Angular Separation
    angular_separation_rad = np.arccos(np.clip(np.dot(shoulder_wrist, shoulder_camera), -1.0, 1.0))
    angular_separation_deg = np.rad2deg(angular_separation_rad)

    # Camera Pitch (Ground Truth / Gyroscope)
    camera_pitch_rad = np.asin(camera_forward_global[-1])
    camera_pitch_deg = np.rad2deg(camera_pitch_rad)

    # Camera Un-Pitch Matrix (Aligns Z-Axis with Gravity)
    c, s = np.cos(-camera_pitch_rad), np.sin(-camera_pitch_rad)
    unpitch_matrix = np.array([
        [c,  0, s],
        [0,  1, 0],
        [-s, 0, c]
    ])

    # Un-Pitch Vectors
    shoulder_wrist_gravity = unpitch_matrix @ shoulder_wrist
    shoulder_wrist_gravity /= np.linalg.norm(shoulder_wrist_gravity)

    shoulder_camera_gravity = unpitch_matrix @ shoulder_camera
    shoulder_camera_gravity /= np.linalg.norm(shoulder_camera_gravity)

    # Yaw & Pitch Components
    shoulder_wrist_gravity_yaw = np.rad2deg(np.atan2(shoulder_wrist_gravity[1], shoulder_wrist_gravity[0]))
    shoulder_wrist_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_wrist_gravity[-1], -1.0, 1.0)))

    shoulder_camera_gravity_yaw = np.rad2deg(np.atan2(shoulder_camera_gravity[1], shoulder_camera_gravity[0]))
    shoulder_camera_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_camera_gravity[-1], -1.0, 1.0)))

    delta_yaw = (shoulder_wrist_gravity_yaw - shoulder_camera_gravity_yaw + 180) % 360 - 180
    delta_pitch = shoulder_wrist_gravity_pitch - shoulder_camera_gravity_pitch

    # Relevant Outputs
    return actor, pose, camera_pitch_deg, distance, angular_separation_deg, delta_yaw, delta_pitch, shoulder_wrist[0], shoulder_wrist[1], shoulder_wrist[2], shoulder_camera[0], shoulder_camera[1], shoulder_camera[2]

In [36]:
# Populate Dataframe

gt_cols = ['gt_dist', 'gt_ang_sep', 'gt_d_yaw', 'gt_d_pitch', 'gt_sw_x', 'gt_sw_y', 'gt_sw_z', 'gt_sc_x', 'gt_sc_y', 'gt_sc_z']

for i in tqdm(range(NUM_ROWS), desc="Calculating GT"):
    try:
        row_id = df.at[i, 'id']
        results = gt_calcs(row_id)
        
        all_cols = ['actor', 'pose', 'cam_pitch'] + gt_cols
        df.loc[i, all_cols] = results

    except FileNotFoundError:
        tqdm.write(f"Missing: {row_id}")

    except Exception as e:
        tqdm.write(f"Error at {row_id}: {e}")

df.head(6)

Calculating GT: 100%|██████████| 4096/4096 [00:05<00:00, 704.86it/s]


,id,actor,pose,cam_pitch,gt_dist,gt_ang_sep,gt_d_yaw,gt_d_pitch,gt_sw_x,gt_sw_y,...,m2pa_dist,m2pa_ang_sep,m2pa_d_yaw,m2pa_d_pitch,m2pa_sw_x,m2pa_sw_y,m2pa_sw_z,m2pa_sc_x,m2pa_sc_y,m2pa_sc_z
0,00001,BP_Ada_C_1,34.0,-6.287881,354.167297,14.391177,-4.068234,13.838668,-0.968621,0.066612,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00002,BP_Ada_C_1,17.0,-80.392527,839.159817,70.426628,10.291927,-70.265796,-0.335014,-0.175880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00003,BP_Ada_C_1,114.0,-50.585528,716.956094,30.322529,-20.436334,29.535145,-0.863197,0.059908,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00004,BP_Ada_C_1,42.0,-20.623707,357.440408,61.804887,-68.575333,9.502579,-0.472476,0.805154,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00005,BP_Ada_C_1,114.0,-16.400097,540.643029,72.647749,82.989366,63.720576,-0.298245,-0.170291,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,00006,BP_Ada_C_1,92.0,-67.072543,511.871710,35.985887,-115.477473,3.051334,-0.809162,0.306925,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
# Save Dataframe

df.to_csv(SAVE_PATH, index=False)

In [38]:
# Sanity Check

check_df = pd.read_csv(SAVE_PATH, dtype={'id': str})
print(check_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4096 entries, 0 to 4095
Data columns (total 84 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            4096 non-null   object 
 1   actor         4096 non-null   object 
 2   pose          4096 non-null   float64
 3   cam_pitch     4096 non-null   float64
 4   gt_dist       4096 non-null   float64
 5   gt_ang_sep    4096 non-null   float64
 6   gt_d_yaw      4096 non-null   float64
 7   gt_d_pitch    4096 non-null   float64
 8   gt_sw_x       4096 non-null   float64
 9   gt_sw_y       4096 non-null   float64
 10  gt_sw_z       4096 non-null   float64
 11  gt_sc_x       4096 non-null   float64
 12  gt_sc_y       4096 non-null   float64
 13  gt_sc_z       4096 non-null   float64
 14  b3ad_dist     0 non-null      float64
 15  b3ad_ang_sep  0 non-null      float64
 16  b3ad_d_yaw    0 non-null      float64
 17  b3ad_d_pitch  0 non-null      float64
 18  b3ad_sw_x     0 non-null    